# RVC 1, Ch 2

Python version of **01_rvc1_ch2.1_2DPos.mlx**, using the Spatial Math Toolbox for Python.

Reference portal: https://pythonrobotics.io/

Current Spatial Math Toolbox repository/documentation: https://github.com/rai-opensource/spatialmath-python

### MATLAB → Python conventions used in this notebook
- MATLAB matrix multiplication `*` becomes NumPy matrix multiplication `@` for raw arrays.
- Spatial Math pose objects such as `SE2` compose with `*`.
- MATLAB `W_T_C.angle` corresponds to Python `W_T_C.theta()`.
- MATLAB `W_T_C.T` (matrix representation) corresponds to Python `W_T_C.A`.
- MATLAB `plotvol(...)` corresponds to Python `plotvol2(...)` for 2D plots.


In [1]:
# Works best in Jupyter Notebook/JupyterLab.
# If needed, install dependencies once in your environment:
# %pip install spatialmath-python sympy matplotlib

%matplotlib notebook

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# Import SE2 class and specific methods from spatialmath base class
from spatialmath import SE2
from spatialmath.base import (
    rot2, trot2, transl2, trplot2, tranimate2,
    plotvol2, plot_point
)

# Close any open plots
plt.close('all')

# Set numpy printing properties
np.set_printoptions(
    linewidth=100,
    formatter={'float': lambda x: f"{x:8.4g}" if abs(x) > 1e-10 else f"{0:8.4g}"}
)


## Lec 01.6 (Ch 2.1) 2D rotations


Let's create a 2D rotation matrix about some angle:


In [2]:
R = rot2(0)
print(R)


[[       1        0]
 [       0        1]]


Let's try a 0.2 radian rotation:


In [3]:
R = rot2(0.2)
print(R)


[[  0.9801  -0.1987]
 [  0.1987   0.9801]]


Rotations can also be defined in degrees. Let's try a 30 degree rotation:


In [4]:
R = rot2(30, 'deg')
print(R)


[[   0.866     -0.5]
 [     0.5    0.866]]


### Rotation Matrix Properties

Rotation matrices are:
1. **orthonormal**: columns are orthogonal to each other and unit length
2. **full rank**: independent/invertible
3. their **inverse equals their transpose**

Let's check them out.


#### A. Orthonormal


Orthonormal vectors both have a norm of 1 and are perpendicular to each other. If we take the dot product of two perpendicular vectors, the result should be 0.


In [5]:
# Extract two columns (MATLAB R(:,1), R(:,2) -> Python zero-based indexing)
c1 = R[:, 0]
c2 = R[:, 1]
print('c1 =', c1)
print('c2 =', c2)


c1 = [   0.866      0.5]
c2 = [    -0.5    0.866]


Compute their norms:


In [6]:
print(np.linalg.norm(c1))
print(np.linalg.norm(c2))


1.0
1.0


Now, let's take their dot product:


In [7]:
print(np.dot(c1, c2))


0.0


#### B. Full Rank


Full rank means that no column or row in the matrix is a linear combination of the others, so the matrix spans the full dimensional space. In two dimensions, the rank of a rotation matrix should be 2.


In [8]:
# Full rank
print(np.linalg.matrix_rank(R))


2


If a matrix is full rank, its determinant will never be zero.


##### Determinant


If the determinant is zero, the matrix is singular (it has dependencies) and cannot span the full space.


In [ ]:
print(np.linalg.det(R))


##### Invertible


Full-rank matrices (those with nonzero determinants) can be inverted. Inverted rotation matrices are inverse rotations. If `R` is associated with a 30 degree rotation, then `inv(R)` is a -30 degree rotation.


In [9]:
R_inv = np.linalg.inv(R)
print(R_inv)


[[   0.866      0.5]
 [    -0.5    0.866]]


Let's compare this with directly forming a -30 degree rotation:


In [10]:
R_minus30 = rot2(-30, 'deg')
print(R_minus30)
print('Equal:', np.allclose(R_inv, R_minus30))


[[   0.866      0.5]
 [    -0.5    0.866]]
Equal: True


### C. Inverse and Transpose

The MATLAB live script labels this section **"Symmetrical Matrices"**. The property actually demonstrated is the orthogonality of a rotation matrix:

\[
R^{-1} = R^T.
\]

A rotation matrix is not, in general, itself a symmetric matrix.


In [11]:
# Transpose
print(R.T)


[[   0.866      0.5]
 [    -0.5    0.866]]


Let's compare with `inv(R)`: 


In [12]:
print(np.linalg.inv(R))
print('inv(R) == R.T:', np.allclose(np.linalg.inv(R), R.T))


[[   0.866      0.5]
 [    -0.5    0.866]]
inv(R) == R.T: True


#### Compounded Rotations (Chains of Rotation Matrices)


Chains of rotation matrices keep the same rotation-matrix properties:


In [13]:
R_compound = R @ R
print(R_compound)
print('det(R @ R) =', np.linalg.det(R_compound))


[[     0.5   -0.866]
 [   0.866      0.5]]
det(R @ R) = 1.0


#### Plotting


In [15]:
# Plot coordinate axes rotated by R with respect to the origin
fig, ax = plt.subplots()
ax.grid(True)
ax.set_aspect('equal')
trplot2(R, ax=ax, dims=[-1, 2, -1, 2])
plt.show()


<IPython.core.display.Javascript object>

#### Symbolic Notation


Python can also work with symbolic notation using SymPy. Spatial Math base rotation functions support symbolic angles when SymPy is installed.


In [16]:
theta = sp.symbols('theta')
R_sym = rot2(theta)
print(R_sym)


[[cos(theta) -sin(theta)]
 [sin(theta) cos(theta)]]


Compounded rotations are also possible:


In [17]:
R2_sym = R_sym @ R_sym
print(R2_sym)


[[-sin(theta)**2 + cos(theta)**2 -2*sin(theta)*cos(theta)]
 [2*sin(theta)*cos(theta) -sin(theta)**2 + cos(theta)**2]]


Complex expressions can sometimes be simplified with `sympy.simplify`:


In [18]:
R2_simplified = np.array(sp.Matrix(R2_sym).applyfunc(sp.simplify), dtype=object)
print(R2_simplified)


[[cos(2*theta) -sin(2*theta)]
 [sin(2*theta) cos(2*theta)]]


Symbolic expressions for the determinant are also obtained:


In [19]:
det_R_sym = sp.Matrix(R_sym).det()
print(det_R_sym)
print(sp.simplify(det_R_sym))


sin(theta)**2 + cos(theta)**2
1


---

## Lec 01.7 (Ch 2.1) 2D Rotation and Translation


In this section we will learn to create translations, along with combinations of translations and rotations (i.e. a homogeneous transform).


We will also learn to distinguish between manual operations on NumPy arrays and the use of the `SE2` class.


### Homogeneous Transformations


Create a homogeneous transformation representation of a pure translation:


In [20]:
T0 = transl2(1, 2)
print(T0)


[[       1        0        1]
 [       0        1        2]
 [       0        0        1]]


Create a homogeneous transformation representation of a pure rotation of 30 degrees, specified in radians or degrees:


In [21]:
R0 = trot2(0.5236)  # 30 degree rotation, approximately, in radians
print(R0)

R0 = trot2(30, 'deg')
print(R0)


[[   0.866     -0.5        0]
 [     0.5    0.866        0]
 [       0        0        1]]
[[   0.866     -0.5        0]
 [     0.5    0.866        0]
 [       0        0        1]]


Let us now combine a 2D homogeneous transformation matrix with a rotation and translation.


We will explore two methods to do this:
1. Manual method with NumPy matrices
2. `SE2` object


#### 1. Manual Homogeneous Transformation Instantiation


In [22]:
# NumPy arrays use @ for matrix multiplication in Python
W_T = transl2(1, 2) @ trot2(30, 'deg')
print(W_T)
print(type(W_T))


[[   0.866     -0.5        1]
 [     0.5    0.866        2]
 [       0        0        1]]
<class 'numpy.ndarray'>


#### 2. Special Euclidean (SE2) Group


The idea of groups comes from **group theory**, an important topic in abstract algebra. Put simply here, an `SE2` object lets us encode rotations and translations while preserving the mathematical structure of the Special Euclidean group.


Note that `SE2` is a class. Calling it instantiates an object of that class.


Let's create a homogeneous transformation that involves a translation of `[1, 2]` and a 30 degree rotation in the 2D plane. This could be the pose of the camera with respect to the world: `W_T_C`.


In [23]:
W_T_C = SE2(1, 2, 30, unit='deg')  # in degrees
print(W_T_C)

W_T_C = SE2(1, 2, 0.5236)  # in radians
print(W_T_C)


   0.866    -0.5       1         
   0.5       0.866     2         
   0         0         1         

   0.866    -0.5       1         
   0.5       0.866     2         
   0         0         1         



Note that this is not a regular NumPy matrix but an `SE2` object:


In [24]:
print(type(W_T_C))
print(isinstance(W_T_C, SE2))


<class 'spatialmath.pose2d.SE2'>
True


You can learn about the methods available to an `SE2` object by calling `help`:


In [ ]:
help(SE2)


You can extract only the rotation matrix:


In [25]:
print(W_T_C.R)


[[   0.866     -0.5]
 [     0.5    0.866]]


You can extract the angle of the rotation. The Python `SE2` API uses `theta()` (the MATLAB live script uses `.angle`):


In [26]:
print(W_T_C.theta())
print(W_T_C.theta(unit='deg'))


0.5236
30.0000701530499


You can extract the translation:


In [27]:
print(W_T_C.t)


[       1        2]


You can extract the homogeneous transformation as a NumPy array. Python uses the `.A` property (instead of the MATLAB `.T` property used in the live script):


In [28]:
print(W_T_C.A)
print(type(W_T_C.A))


[[   0.866     -0.5        1]
 [     0.5    0.866        2]
 [       0        0        1]]
<class 'numpy.ndarray'>


You can plot the transformation:


In [29]:
W_T_C.plot(frame='W_T_C', color='blue', dims=[0, 5, -2, 5])
plt.show()


<IPython.core.display.Javascript object>

Better yet, you can animate the transformation from the original frame to the current frame. This works best with an interactive Matplotlib backend:


In [ ]:
# Run this cell interactively to animate:
W_T_C.animate(frame='W_T_C', dims=[0, 5, -2, 5])


If you do not have an `SE2` object, you can still plot a regular homogeneous transformation using `trplot2`:


In [ ]:
# As in the MATLAB live script, redefine W_T_C here as a raw homogeneous matrix.
# Note that this version is a pure translation (no 30-degree rotation).
W_T_C = transl2(1, 2)

ax = plotvol2([0, 5, -2, 5], grid=True, new=True)
trplot2(W_T_C, frame='W_T_C', color='blue', ax=ax)
plt.show()


Note: MATLAB graphics properties often appear as `'property_name', 'property_value'` pairs. Python uses keyword arguments such as `frame='W_T_C'` and `color='blue'`.


You can adjust the axis with `plotvol2`:


In [ ]:
ax = plotvol2([0, 5, -2, 5], grid=True, new=True)
trplot2(W_T_C, frame='W_T_C', color='blue', ax=ax)
plt.show()


### Compound Pose Transformations


Consider a second pose, one that identifies the robot with respect to the world: `W_T_R`.


In [ ]:
W_T_R = transl2(2, 1)
print(W_T_R)


Visualize both poses:


In [ ]:
ax = plotvol2([0, 5, -2, 5], grid=True, new=True)
trplot2(W_T_C, frame='W_T_C', color='blue', ax=ax)
trplot2(W_T_R, frame='W_T_R', color='red', ax=ax)
plt.show()


What if we want to get the robot pose with respect to the camera: `C_T_R`?

To get there we need

\[
{}^C T_R = {}^C T_W\,{}^W T_R
\]

and

\[
{}^C T_W = ({}^W T_C)^{-1}.
\]


In [ ]:
C_T_R = np.linalg.inv(W_T_C) @ W_T_R
print(C_T_R)


### Transforming Vectors


Let us now consider a point `P` (imagine the center of mass of a target object). Its coordinates are with respect to one frame of reference, but we may want to transform that point into another frame used by the robot arm.


In [ ]:
# Create a point P that is [3, 2] with respect to the camera frame.
# Use a homogeneous coordinate so it can be multiplied by a 3x3 SE(2) matrix.
C_P = np.array([3.0, 2.0, 1.0])
print(C_P)


To express the point with respect to the world frame, compute

\[
{}^W P = {}^W T_C\,{}^C P.
\]

The MATLAB live script assigns this result back into `C_P`; here it is stored as `W_P` so the variable name matches the frame notation in the accompanying text.


In [ ]:
W_P = W_T_C @ C_P
print(W_P)


To plot the point, extract the first two coordinates:


In [ ]:
ax = plotvol2([0, 6, -2, 6], grid=True, new=True)
trplot2(W_T_C, frame='W_T_C', color='blue', ax=ax)
trplot2(W_T_R, frame='W_T_R', color='red', ax=ax)
plot_point(W_P[:2], marker='ko', text='W_P', textcolor='k', ax=ax)
plt.show()


What might the original camera-frame point `C_P = [3, 2]` be with respect to the robot frame `R_P`?

1. Get the transformation of the camera with respect to the robot: `R_T_C`.
2. Compute `R_P = R_T_C @ C_P`.
3. To get `R_T_C`, use `R_T_W @ W_T_C`, where `R_T_W = inv(W_T_R)`.


In [ ]:
# Get R_T_W, then R_T_C
R_T_C = np.linalg.inv(W_T_R) @ W_T_C
print(R_T_C)

# Note: unlike a pure rotation matrix, the inverse of a homogeneous
# transformation matrix is not generally equal to its transpose.
print('inv(W_T_C) equals W_T_C.T:', np.allclose(np.linalg.inv(W_T_C), W_T_C.T))


Let's do one more coordinate transform, this time from the camera coordinate system to the robot coordinate system.


Consider the original coordinates of our target object with respect to the camera frame, `C_P = [3, 2]`. We want `R_P = R_T_C @ C_P`. What coordinate would you expect to get? `[2, 3]`? Do we get it?


In [ ]:
C_P = np.array([3.0, 2.0, 1.0])
R_P = R_T_C @ C_P
print(R_P)
print('Euclidean coordinates:', R_P[:2])


---
### Summary of key Python equivalents

| MATLAB live script | Python Spatial Math / NumPy |
|---|---|
| `R = rot2(theta)` | `R = rot2(theta)` |
| `R(:,1)` | `R[:, 0]` |
| `rank(R)` | `np.linalg.matrix_rank(R)` |
| `det(R)` | `np.linalg.det(R)` |
| `inv(R)` | `np.linalg.inv(R)` |
| `R'` | `R.T` |
| `R*R` | `R @ R` for NumPy arrays |
| `transl2(1,2)` | `transl2(1, 2)` |
| `trot2(30,'deg')` | `trot2(30, 'deg')` |
| `SE2(1,2,30,'deg')` | `SE2(1, 2, 30, unit='deg')` |
| `W_T_C.angle` | `W_T_C.theta()` |
| `W_T_C.t` | `W_T_C.t` |
| `W_T_C.R` | `W_T_C.R` |
| `W_T_C.T` | `W_T_C.A` |
| `W_T_C.plot` | `W_T_C.plot(...)` |
| `W_T_C.animate` | `W_T_C.animate(...)` |
| `plotvol(...)` | `plotvol2(...)` |
| `plot_point(...)` | `plot_point(...)` with Python keyword arguments |
